# 問題
以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。
- “The movie was full of fun.”
- “The movie was full of excitement.”
- “The movie was full of crap.”
- “The movie was full of rubbish.”

In [2]:
from transformers import AutoTokenizer

model_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
def tokenize(text):
  inputs = tokenizer(text, return_tensors="pt")
  return inputs

text1 = "The movie was full of fun."
text2 = "The movie was full of excitement."
text3 = "The movie was full of crap."
text4 = "The movie was full of rubbish."
token1 = tokenize(text1)
token2 = tokenize(text2)
token3 = tokenize(text3)
token4 = tokenize(text4)

from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(model_id)

def get_mean_embedding(inputs, exclude_special=True):
    with torch.no_grad():
        outputs = model(**inputs)  # last_hidden_state: [1, seq_len, hidden]
    hidden = outputs.last_hidden_state           # [1, L, H]
    mask = inputs["attention_mask"]             # [1, L] (1=有効, 0=pad)

    if exclude_special:
        ids = inputs["input_ids"]               # [1, L]
        special = (
            (ids == tokenizer.cls_token_id) |
            (ids == tokenizer.sep_token_id) |
            (ids == tokenizer.pad_token_id)
        )
        # special トークンは平均から除外
        mask = mask * (~special).int()

    # マスクをかけて総和→有効トークン数で割る
    masked = hidden * mask.unsqueeze(-1)        # [1, L, H]
    sum_vec = masked.sum(dim=1)                 # [1, H]
    count = mask.sum(dim=1).clamp(min=1)        # [1]
    mean_vec = sum_vec / count.unsqueeze(-1)    # [1, H]
    return mean_vec[0]  
                            # [H]
import torch.nn.functional as F
emb1 = get_mean_embedding(token1)
emb2 = get_mean_embedding(token2)
emb3 = get_mean_embedding(token3)
emb4 = get_mean_embedding(token4)

texts = [text1, text2, text3, text4]
embs  = [emb1, emb2, emb3, emb4]

for i in range(len(embs)):
    for j in range(i+1, len(embs)):
        sim = F.cosine_similarity(embs[i], embs[j], dim=0)
        print(f"{texts[i]} vs {texts[j]} -> {sim.item():.4f}")

The movie was full of fun. vs The movie was full of excitement. -> 0.9535
The movie was full of fun. vs The movie was full of crap. -> 0.8379
The movie was full of fun. vs The movie was full of rubbish. -> 0.8046
The movie was full of excitement. vs The movie was full of crap. -> 0.8256
The movie was full of excitement. vs The movie was full of rubbish. -> 0.7849
The movie was full of crap. vs The movie was full of rubbish. -> 0.9182


トークンごとのベクトル（hidden）: [CLS], The, movie, was, full, of, fun, ., [SEP]
マスク（mask）:                    [0,    1,    1,    1,   1,   1,   1, 1,  0]

① maskを掛ける         → [0, v2, v3, v4, v5, v6, v7, v8, 0]<br>
② 足し算（sum）        → v2 + v3 + v4 + v5 + v6 + v7 + v8<br>
③ トークン数で割る     → (v2 + v3 + ... + v8) / 7<br>
④ 平均ベクトル完成     → 文全体の意味ベクトル（mean pooling）